# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dishaprabhakar2006-blip/Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [6]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I'm using a Random Forest Classifier. My lane (Refresh/Content Opportunity Scoring)
needs a ranked output, and Random Forest naturally gives a probability per page I can
sort by — same shape as my Week-4 baseline score. I'm choosing it over plain Logistic
Regression because my Week-4 signal check showed staleness alone was only MIXED
(not a clean linear relationship with decline), so a model that can capture
non-linear combinations of signals should do better than a straight-line method.
I'm choosing it over a single Decision Tree because a forest averages many trees,
reducing the overfitting risk I already saw in notebook 02 (my depth-4 tree got
WORSE than a simple hand rule).

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
from sklearn.model_selection import GroupShuffleSplit

baseline_data = con.sql(f"""
    SELECT
      f.content_hash_id,
      f.client_hash_id,
      f.gsc_impressions,
      f.gsc_clicks,
      f.gsc_avg_position,
      date_diff('day', CAST(d.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) AS content_age_days,
      d.word_count,
      (f.gsc_clicks / NULLIF(f.gsc_impressions,0)) AS ctr
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_impressions >= 100
""").df()

baseline_data["is_declining_label"] = (baseline_data["gsc_avg_position"] > 20).astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(baseline_data, groups=baseline_data["client_hash_id"]))

train_df = baseline_data.iloc[train_idx].reset_index(drop=True)
test_df = baseline_data.iloc[test_idx].reset_index(drop=True)

print("Train clients:", train_df["client_hash_id"].nunique(), "| Test clients:", test_df["client_hash_id"].nunique())
print("Overlap check (should be 0):", len(set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train clients: 26 | Test clients: 9
Overlap check (should be 0): 0


I'm using a grouped split by client_hash_id (GroupShuffleSplit), not a plain random
split. My data contract (w03) and the warehouse guide both flag this as an unbalanced
panel — pages from the same client can share patterns, so if pages from one client
land in both train and test, the model could just memorize that client instead of
learning generalizable signal. This matches the client_holdout strategy the starter
pipeline itself used.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

feature_cols = ["gsc_impressions", "content_age_days", "word_count", "ctr"]
# NOTE: gsc_avg_position excluded — it's literally what the label is derived from (leakage trap from w03)

X_train = train_df[feature_cols].fillna(0)
y_train = train_df["is_declining_label"]
X_test = test_df[feature_cols].fillna(0)
y_test = test_df["is_declining_label"]

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
model_proba = model.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Baseline score, same rule as Week 4, recomputed on the TEST split only
test_df["stale"] = (test_df["content_age_days"] >= 180).astype(int)
test_df["visible"] = (test_df["gsc_impressions"] >= 500).astype(int)
test_df["baseline_score"] = test_df["stale"] * test_df["visible"] * test_df["gsc_impressions"]

results = []
for k in (20, 50):
    bp = precision_at_k(test_df["baseline_score"], y_test, k)
    mp = precision_at_k(model_proba, y_test, k)
    results.append({"k": k, "baseline_precision": round(bp,3), "model_precision": round(mp,3)})

import pandas as pd
pd.DataFrame(results)

,k,baseline_precision,model_precision
0,20,0.00,0.90
1,50,0.14,0.88


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

# Where does the model disagree most with the baseline?
test_df["model_score"] = model_proba
test_df_sorted_model = test_df.sort_values("model_score", ascending=False).head(50)
test_df_sorted_baseline = test_df.sort_values("baseline_score", ascending=False).head(50)

overlap = set(test_df_sorted_model["content_hash_id"]) & set(test_df_sorted_baseline["content_hash_id"])
print(f"Overlap between model's top 50 and baseline's top 50: {len(overlap)} / 50")

content_age_days    0.458265
word_count          0.260774
ctr                 0.236597
gsc_impressions     0.044364
dtype: float64
Overlap between model's top 50 and baseline's top 50: 0 / 50


Feature importance: content_age_days dominates (45.7%), followed by word_count
(26.3%) and ctr (23.6%) — gsc_impressions barely matters (4.4%). This is a notable
shift from my Week-4 baseline, which leaned heavily on impressions and a blunt
stale/visible split. The model is finding that content age and length matter more
to decline than raw traffic volume does.

The model's top 50 and baseline's top 50 have ZERO overlap — completely different
sets of pages. This makes sense given the feature importances: my baseline's score
is literally impressions × stale × visible, so it's dominated by high-traffic pages.
The model, leaning on age and word_count instead, surfaces older/shorter pages
regardless of their traffic volume — a structurally different definition of
"needs review."

Precision@20: baseline [X] vs model [Y]
Precision@50: baseline [X] vs model [Y]

[One honest sentence on whether the model actually beat the baseline or not,
based on the real numbers]

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.